# Olist E-commerce Performance AnalysisEnd-to-end analysis of the Brazilian E-Commerce Public Dataset by Olist using Python, SQL/SQLite, and Power BI.## Run locally1. Install dependencies from `requirements.txt`.2. Place `olist.db` in the same folder as this notebook.3. Run all cells from top to bottom.Product revenue is calculated from item price and excludes freight. Delivery/review relationships are associations, not proof of causation. Product frequency counts order-item rows, not distinct orders.

## 1. Load the database and inspect data qualityLoad the eight SQLite tables, parse date columns, and inspect table sizes, missing values, and duplicate rows.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = Path("olist.db")
if not DB_PATH.exists():
    raise FileNotFoundError("Place olist.db beside this notebook before running it.")

with sqlite3.connect(DB_PATH) as conn:
    orders = pd.read_sql_query("SELECT * FROM orders", conn)
    customers = pd.read_sql_query("SELECT * FROM customers", conn)
    items = pd.read_sql_query("SELECT * FROM order_items", conn)
    payments = pd.read_sql_query("SELECT * FROM payments", conn)
    reviews = pd.read_sql_query("SELECT * FROM order_reviews", conn)
    products = pd.read_sql_query("SELECT * FROM products", conn)
    sellers = pd.read_sql_query("SELECT * FROM sellers", conn)
    translation = pd.read_sql_query("SELECT * FROM translation", conn)

date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

tables = {
    "orders": orders, "customers": customers, "order_items": items,
    "payments": payments, "order_reviews": reviews, "products": products,
    "sellers": sellers, "translation": translation,
}
for name, frame in tables.items():
    print(f"{name}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")
    print("Missing values:\n", frame.isna().sum())
    print("Duplicate rows:", frame.duplicated().sum(), "\n")


## 2. Delivery performance analysisMeasure delivery duration and late-delivery rates, then compare delivery performance across customer states.

In [ ]:
# Delivered-order and delivery performance analysis
delivered = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
].copy()
delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]
).dt.total_seconds() / 86400
delivered["is_late"] = (
    delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
)
print("Delivered orders:", len(delivered))
print("Average delivery days:", round(delivered["delivery_days"].mean(), 2))
print("Median delivery days:", round(delivered["delivery_days"].median(), 2))
print("Late delivery rate (%):", round(delivered["is_late"].mean() * 100, 2))

delivery = delivered.merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left"
)
by_state = delivery.groupby("customer_state").agg(
    orders=("order_id", "count"),
    avg_days=("delivery_days", "mean"),
    late_pct=("is_late", "mean"),
)
by_state["late_pct"] *= 100
by_state = by_state[by_state["orders"] >= 100]
print("\nSlowest states (100+ delivered orders):")
print(by_state.sort_values("avg_days", ascending=False).head(10).round(2))


## 3. Delivery timeliness and customer reviewsCompare average review scores for late and on-time deliveries. This is an association, not evidence of causation.

In [ ]:
# Relationship between delivery timeliness and customer review scores
score = reviews.groupby("order_id")["review_score"].mean().reset_index()
review_delivery = delivery.merge(score, on="order_id", how="inner")
review_summary = review_delivery.groupby("is_late")["review_score"].agg(["count", "mean"])
print("\nReview score by delivery status:")
print(review_summary.round(2))


## 4. Sales, categories, and monthly revenueCalculate delivered-order product revenue (excluding freight), export summary tables, and create the main revenue and delivery charts.

In [ ]:
# Delivered sales, categories and monthly revenue (item price excludes freight)
sales = items.merge(
    orders.loc[orders["order_status"] == "delivered",
               ["order_id", "customer_id", "order_purchase_timestamp"]],
    on="order_id",
)
sales = sales.merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
sales = sales.merge(translation, on="product_category_name", how="left")
sales["category"] = (
    sales["product_category_name_english"]
    .fillna(sales["product_category_name"])
    .fillna("unknown")
)
sales["month"] = sales["order_purchase_timestamp"].dt.to_period("M")
product_revenue_total = sales["price"].sum()
delivered_order_count = sales["order_id"].nunique()
print("\nProduct revenue (BRL, excludes freight):", round(product_revenue_total, 2))
print("Delivered-order AOV (BRL):", round(product_revenue_total / delivered_order_count, 2))

monthly = sales.groupby("month")["price"].sum()
monthly.index = monthly.index.astype(str)
monthly = monthly.loc["2017-01":"2018-08"]
monthly.rename_axis("month").reset_index(name="revenue").to_csv(
    "monthly_revenue.csv", index=False
)
plt.figure(figsize=(10, 4))
plt.plot(monthly.index, monthly.values, marker="o")
plt.xticks(rotation=90)
plt.title("Monthly product revenue (delivered orders)")
plt.ylabel("Revenue (BRL)")
plt.tight_layout()
plt.savefig("monthly_revenue.png", dpi=150)
plt.show()

category_revenue = sales.groupby("category")["price"].sum().sort_values(ascending=False)
category_revenue.rename_axis("category").reset_index(name="revenue").to_csv(
    "revenue_by_category.csv", index=False
)
top_categories = category_revenue.head(10)
plt.figure(figsize=(8, 5))
plt.barh(top_categories.index[::-1], top_categories.values[::-1])
plt.title("Top 10 product categories by revenue")
plt.tight_layout()
plt.savefig("top_categories.png", dpi=150)
plt.show()

by_state.reset_index().to_csv("delivery_by_state.csv", index=False)
slowest = by_state.sort_values("avg_days", ascending=False).head(10)
plt.figure(figsize=(8, 4))
plt.bar(slowest.index, slowest["avg_days"])
plt.title("Average delivery days: 10 slowest states")
plt.ylabel("Days")
plt.tight_layout()
plt.savefig("slowest_states.png", dpi=150)
plt.show()

review_means = review_delivery.groupby("is_late")["review_score"].mean()
plt.figure(figsize=(5, 4))
plt.bar(["On time", "Late"], [review_means[False], review_means[True]])
plt.ylim(0, 5)
plt.title("Average review score: on time vs late")
plt.tight_layout()
plt.savefig("score_late_vs_ontime.png", dpi=150)
plt.show()


## 6. Seller performanceCompare seller-level late-delivery rates with average review scores, using sellers with at least 30 eligible orders.

## 5. SQL validationCross-check the main revenue and customer-review metrics with SQL. The validation tables are loaded into an in-memory SQLite database, so the original `olist.db` file is never overwritten.

In [ ]:
sql_conn = sqlite3.connect(":memory:")
for table_name, frame in tables.items():
    frame.to_sql(table_name, sql_conn, index=False, if_exists="replace")

sql_revenue_check = pd.read_sql_query("""
SELECT
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    ROUND(SUM(i.price), 2) AS product_revenue,
    ROUND(SUM(i.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
""", sql_conn)
print("SQL revenue validation:")
print(sql_revenue_check)

sql_review_check = pd.read_sql_query("""
WITH order_reviews_avg AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
)
SELECT
    CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
         THEN 'Late' ELSE 'On time' END AS delivery_status,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM orders o
JOIN order_reviews_avg r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY delivery_status
ORDER BY delivery_status
""", sql_conn)
print("\nSQL review validation:")
print(sql_review_check)
sql_conn.close()


In [ ]:
# Seller performance; only sellers with at least 30 eligible orders in the comparisonseller_orders = items[["order_id", "seller_id"]].drop_duplicates()seller_orders = seller_orders.merge(    delivered[["order_id", "is_late"]], on="order_id", how="inner").merge(score, on="order_id", how="left")seller_perf = seller_orders.groupby("seller_id").agg(    orders=("order_id", "count"),    late_orders=("is_late", "sum"),    avg_score=("review_score", "mean"),)seller_perf["late_pct"] = seller_perf["late_orders"] / seller_perf["orders"] * 100seller_revenue = sales.merge(    delivered[["order_id"]], on="order_id", how="inner").groupby("seller_id")["price"].sum()seller_perf = seller_perf.join(seller_revenue.rename("revenue"))seller_perf = seller_perf[seller_perf["orders"] >= 30]seller_perf.reset_index().to_csv("seller_performance.csv", index=False)print("\nSeller late-rate / review-score correlation:",      round(seller_perf["late_pct"].corr(seller_perf["avg_score"]), 3))plt.figure(figsize=(7, 5))plt.scatter(seller_perf["late_pct"], seller_perf["avg_score"], alpha=0.5)plt.xlabel("Late orders (%)")plt.ylabel("Average review score")plt.title("Sellers with 30+ orders: late deliveries vs review score")plt.tight_layout()plt.savefig("seller_late_vs_score.png", dpi=150)plt.show()

## 7. Customer, product, and payment insightsExplore repeat purchasing, product frequency, revenue concentration, and credit-card installment behaviour.

In [ ]:
# Customer repeat purchasing among delivered orders
delivered_customers = delivered[["customer_id"]].merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left"
)
customer_order_counts = delivered_customers.groupby("customer_unique_id")[
    "customer_id"
].nunique()
print("\nUnique delivered customers:", customer_order_counts.size)
print("Customers with one delivered order:", int((customer_order_counts == 1).sum()))
print("Customers with 2+ delivered orders:", int((customer_order_counts >= 2).sum()))
print("Repeat-customer share (%):",
      round((customer_order_counts >= 2).mean() * 100, 2))

# Product frequency counts order-item rows, not distinct orders
product_item_counts = sales.groupby("product_id").size()
product_revenue = sales.groupby("product_id")["price"].sum().sort_values(ascending=False)
top_n = int(len(product_revenue) * 0.01)
print("\nDistinct products in delivered orders:", len(product_item_counts))
print("Products appearing in exactly one order-item row (%):",
      round((product_item_counts == 1).mean() * 100, 2))
print("Top 1% product share of product revenue (%):",
      round(product_revenue.head(top_n).sum() / product_revenue.sum() * 100, 2))

# Installment usage among credit-card payment records across all order statuses
credit_card = payments[payments["payment_type"] == "credit_card"]
print("\nCredit-card payment records:", len(credit_card))
print("Credit-card records with 2+ installments (%):",
      round((credit_card["payment_installments"] > 1).mean() * 100, 2))
print("Average credit-card installments:",
      round(credit_card["payment_installments"].mean(), 2))


## 8. Prepare Power BI fact tablesCreate order-level and order-item-level tables for the Power BI model and export them as CSV files.

In [ ]:
# Fact tables for the Power BI data model
order_revenue = items.groupby("order_id")["price"].sum().rename("revenue")
orders_fact = orders.loc[
    orders["order_status"] == "delivered",
    ["order_id", "customer_id", "order_purchase_timestamp",
     "order_delivered_customer_date", "order_estimated_delivery_date"],
].copy()
orders_fact = orders_fact.merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left"
).merge(order_revenue, on="order_id", how="left").merge(
    score, on="order_id", how="left"
)
orders_fact["purchase_date"] = orders_fact["order_purchase_timestamp"].dt.date
orders_fact["month"] = orders_fact["order_purchase_timestamp"].dt.strftime("%Y-%m")
orders_fact["delivery_days"] = (
    orders_fact["order_delivered_customer_date"]
    - orders_fact["order_purchase_timestamp"]
).dt.total_seconds() / 86400
orders_fact["is_late"] = (
    orders_fact["order_delivered_customer_date"]
    > orders_fact["order_estimated_delivery_date"]
).astype(float)
orders_fact["is_late"] = orders_fact["is_late"].where(
    orders_fact["order_delivered_customer_date"].notna()
)
orders_fact = orders_fact[
    ["order_id", "purchase_date", "month", "customer_state", "revenue",
     "delivery_days", "is_late", "review_score"]
]
items_fact = sales[["order_id", "seller_id", "category", "price"]].copy()
orders_fact.to_csv("orders_fact.csv", index=False)
items_fact.to_csv("items_fact.csv", index=False)
print("\nExported fact tables:", orders_fact.shape, items_fact.shape)
